In [ ]:
import os
import re
import time
from pathlib import Path
import pandas as pd
import requests

# --- 1. REGEX PATTERNS FOR PARAMETER EXTRACTION ---
PARAM_PATTERNS = {
    "argon2": r"\$argon2(?:id|i|d)\$v=\d+\$m=(\d+),t=(\d+),p=(\d+)",
    "bcrypt": r"\$2[abxy]\$(\d{2})\$",
    "pbkdf2_func": r"pbkdf2_hmac\([^,]+,\s*[^,]+,\s*[^,]+,\s*(\d+)",
    "scrypt_func": r"scrypt\([^,]+,\s*[^,]+,\s*(?:n=)?(\d+),\s*(?:r=)?(\d+),\s*(?:p=)?(\d+)",
    "yescrypt_str": r"\$yescrypt\$v=\d+\$m=(\d+),t=(\d+),p=(\d+)",
    "yescrypt_func": r"yescrypt(?:\w*)?\([^,]+,\s*[^,]+,\s*(?:N=)?(\d+),\s*(?:r=)?(\d+),\s*(?:p=)?(\d+)",
    "lyra2_str": r"\$lyra2\$v=\d+\$m=(\d+),t=(\d+),p=(\d+)",
    "lyra2_func": r"lyra2(?:\w*)?\([^,]+,\s*[^,]+,\s*(?:timeCost=)?(\d+),\s*(?:nRows=)?(\d+),\s*(?:nCols=)?(\d+)",
}


def convert_to_raw_url(url: str) -> str:
    """Converts standard GitHub file view URLs to raw downloadable file URLs."""
    if "github.com" in url and "/blob/" in url:
        return url.replace("github.com", "raw.githubusercontent.com").replace(
            "/blob/", "/"
        )
    return url


def parse_parameters(code_text: str, algorithm: str):
    matching_keys = [
        k for k in PARAM_PATTERNS.keys() if algorithm.lower() in k.lower()
    ]
    for key in matching_keys:
        match = re.search(PARAM_PATTERNS[key], str(code_text), re.IGNORECASE)
        if match:
            return match.groups()
    return None


# --- 2. AUTO-LOCATE CSV FILES ---
# Searches current directory and subdirectories automatically
all_csvs = list(Path(".").rglob("*.csv"))

print(f"Directory: {os.getcwd()}")
print(f"Total CSV files found: {len(all_csvs)}")

if len(all_csvs) == 0:
    print(
        " Still found 0 CSVs. Make sure your script is saved inside the root dataset folder"
    )
    exit()

# Target algorithm keywords (partial match allows 'argon2', 'argon2id', 'pbkdf', 'pbkdf2', etc.)
TARGET_ALGOS = [
    "argon2",
    "bcrypt",
    "scrypt",
    "pbkdf",
    "yescrypt",
    "lyra2",
]

extracted_data = []

# --- 3. CRAWL FILES AND LINKS ---
for csv_path in all_csvs:
    algorithm = csv_path.parent.name
    language = csv_path.parent.parent.name
    tier = csv_path.stem

    # Broad match check so folder naming differences don't skip files
    if not any(algo_key in algorithm.lower() for algo_key in TARGET_ALGOS):
        continue

    try:
        df = pd.read_csv(csv_path)

        # Locate URL column regardless of case or whitespace ('url', 'URL', 'url ')
        url_col = None
        for col in df.columns:
            if str(col).strip().lower() == "url":
                url_col = col
                break

        if url_col is None:
            print(
                f"Skipping {csv_path.name}: 'url' column not found in {df.columns.tolist()}"
            )
            continue

        print(
            f"Processing: {language}/{algorithm}/{tier}.csv ({len(df)} rows)..."
        )

        for idx, row in df.iterrows():
            raw_link = str(row[url_col]).strip()
            if not raw_link.startswith("http"):
                continue

            target_url = convert_to_raw_url(raw_link)

            try:
                res = requests.get(
                    target_url,
                    headers={"User-Agent": "MSc-Research-Scanner"},
                    timeout=10,
                )

                if res.status_code == 200:
                    params = parse_parameters(res.text, algorithm)
                    if params:
                        extracted_data.append(
                            {
                                "language": language,
                                "algorithm": algorithm,
                                "tier": tier,
                                "parameters": params,
                                "url": raw_link,
                            }
                        )
                elif res.status_code == 429:
                    print("Rate limit encountered. Sleeping 10s...")
                    time.sleep(10)

            except Exception:
                pass

            time.sleep(0.05)

    except Exception as e:
        print(f"Error reading {csv_path}: {e}")

# --- 4. OUTPUT RESULTS ---
results_df = pd.DataFrame(extracted_data)
results_df.to_csv("extracted_parameters_from_links.csv", index=False)
print(f"\nDone! Successfully extracted {len(results_df)} parameters.")

Directory: /Users/esmewright/Desktop/Cybersecurity MSc/Dissertation/utils
Total CSV files found: 1

Done! Successfully extracted 0 parameters.
